# Audio Transcription Pipeline

In [ ]:
# Install necessary dependencies 
!apt-get update -qq && apt-get install -y ffmpeg
!pip install -q faster-whisper autocorrect torchaudio

In [ ]:
import os
import torchaudio
from faster_whisper import WhisperModel,  BatchedInferencePipeline
from autocorrect import Speller

**INSERT FILE NAME:**

In [ ]:
INPUT_AUDIO = "..."  

In [ ]:
SAMPLING_RATE = 16000

if not os.path.exists(INPUT_AUDIO):
    raise FileNotFoundError(f"\n\n\u274c INPUT FILE MISSING: Please upload an audio file named '{INPUT_AUDIO}' to your Kaggle working directory before running this cell!")

# Temporary files 
WAV_AUDIO = "step1_trimmed.wav"
SILCUT_AUDIO = "step2_silcut.wav"
CPS_AUDIO = "step3_norm.wav"


### 1. Preprocessing (FFmpeg)

In [ ]:
print("--- 1. PREPROCESSING ---")

# 1. WAV Conversion
print("Converting full audio to WAV...")
cmd1 = f'ffmpeg -y -i "{INPUT_AUDIO}" -c:a pcm_s16le -ar {SAMPLING_RATE} "{WAV_AUDIO}" > "{WAV_AUDIO}.log" 2>&1'
if os.system(cmd1) != 0:
    raise RuntimeError(f"\u274c FFmpeg failed at Step 1. Please check the '{WAV_AUDIO}.log' file for details.")

# 2. Silence Removal & Volume Normalization
print("Removing silence...")
cmd2 = f'ffmpeg -y -i "{WAV_AUDIO}" -af "silenceremove=start_periods=1:stop_periods=-1:start_threshold=-50dB:stop_threshold=-50dB:start_silence=0.2:stop_silence=0.2, loudnorm" -c:a pcm_s16le -ar {SAMPLING_RATE} "{SILCUT_AUDIO}" > "{SILCUT_AUDIO}.log" 2>&1'
if os.system(cmd2) != 0:
    raise RuntimeError(f"\u274c FFmpeg failed at Step 2. Please check the '{SILCUT_AUDIO}.log' file for details.")

# 3. Speech Normalization & WAV Conversion
print("Normalizing speech...")
cmd3 = f'ffmpeg -y -i "{SILCUT_AUDIO}" -af "speechnorm=e=50:r=0.0005:l=1" -c:a pcm_s16le -ar {SAMPLING_RATE} "{CPS_AUDIO}" > "{CPS_AUDIO}.log" 2>&1'
if os.system(cmd3) != 0:
    raise RuntimeError(f"\u274c FFmpeg failed at Step 3. Please check the '{CPS_AUDIO}.log' file for details.\n(Note: Your FFmpeg version might not support the 'speechnorm' filter)")

print("Preprocessing complete! Output saved to:", CPS_AUDIO)

### 2. Transcription (Faster Whisper)

In [ ]:
print("--- 2. TRANSCRIPTION ---")

model_size = "large-v3"

model = WhisperModel(model_size, device="cuda", compute_type="float16")
batched_model = BatchedInferencePipeline(model=model)

segments_gen, info = batched_model.transcribe(
    CPS_AUDIO, 
    language="it",
    beam_size=5, 
    vad_filter=True,
    vad_parameters=dict(min_silence_duration_ms=500),
    batch_size=16,
)

# Convert generator to list so we can iterate over it in the postprocessing block
segments = list(segments_gen)

for segment in segments:
    print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))

### 3. Postprocessing

In [ ]:
print("--- 3. POSTPROCESSING ---")

def remove_non_ascii_en(text):
    return ''.join(i for i in text if ord(i)<128)

def remove_non_ascii_it(text):
    # Set of both grave and acute accents used in Italian, upper and lowercase
    italian_accents = set("àèéìíòóùúÀÈÉÌÍÒÓÙÚ")
    return ''.join(i for i in text if ord(i) < 128 or i in italian_accents)

# Initialize the lightweight spell checker for Italian
spell = Speller(lang='it')

processed_transcript = []

for segment in segments:
    raw_text = segment.text
    
    # 1. Remove non-ASCII (Using Italian function)
    clean_text = remove_non_ascii_it(raw_text)
    
    # 2. Correct Misspellings
    corrected_text = spell(clean_text)
    
    processed_transcript.append({
        "start": segment.start,
        "end": segment.end,
        "text": corrected_text
    })
    
    print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, corrected_text))
